## 01 確認 A100、PyTorch 與磁碟
看到 CUDA True、A100 約 79.3 GiB、足夠的可用磁碟。

In [ ]:
import torch, shutil, sys
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print('Free disk (GiB):', round(shutil.disk_usage('/content').free / 1024**3, 1))

## 02 安裝 ComfyUI
若 /content/ComfyUI 已存在就跳過 Git clone；安裝 ComfyUI 基礎相依。

In [ ]:
import subprocess, sys
from pathlib import Path
COMFY = Path('/content/ComfyUI')
if not (COMFY / 'main.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY)], check=True)
else:
    print('ComfyUI already exists. Skipping clone.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(COMFY / 'requirements.txt')], check=True)
for name in ['diffusion_models', 'text_encoders', 'vae', 'loras', 'embeddings']:
    (COMFY / 'models' / name).mkdir(parents=True, exist_ok=True)
print('ComfyUI ready:', COMFY)

## 03 安裝 Hugging Face 下載套件
避免直接無界限升級 huggingface_hub；若 pip check 有衝突先修復再繼續。

In [ ]:
# Avoid `pip install -U huggingface_hub` without a version bound:
# hub >= 2 caused conflicts in the user's Colab environment.
%pip install 'huggingface_hub>=0.34,<2' hf_xet jedi
%pip check

## 04 下載 H3 所需的五個檔案
包含 INT8 FL2VA、文字編碼器、INT8 Video VAE、Audio VAE、8-step Turbo LoRA。若檔案已存在會跳過。

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files
REPO = 'Comfy-Org/MiniMax-H3'
BASE = Path('/content/ComfyUI/models')
# The four core files plus two workflow-specific files observed in the H3 I2V template.
FILES = [
    'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors',
    'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
    'vae/minimax_h3_video_vae_int8_convrot.safetensors',
    'vae/minimax_h3_audio_vae_fp32.safetensors',
    'loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors',
]
remote_files = set(list_repo_files(REPO))
missing_on_hub = [p for p in FILES if p not in remote_files]
if missing_on_hub:
    raise RuntimeError('Names not found in upstream repo: '+', '.join(missing_on_hub))
for name in FILES:
    path = BASE / name
    if path.exists() and path.stat().st_size > 0:
        print('Exists (skipping):', name)
        continue
    print('Downloading:', name)
    out = hf_hub_download(repo_id=REPO, filename=name, local_dir=str(BASE))
    print('Saved:', out)
print('Required model downloads finished.')

## 05 檢查模型位置與大小
確認五個檔案存在；如果模板提示缺少，先對照實際完整檔名。

In [ ]:
from pathlib import Path
base=Path('/content/ComfyUI/models')
required=[
'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors',
'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
'vae/minimax_h3_video_vae_int8_convrot.safetensors',
'vae/minimax_h3_audio_vae_fp32.safetensors',
'loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors']
ok=True
for name in required:
    p=base/name
    status=p.is_file() and p.stat().st_size>0
    ok &= status
    print('OK' if status else 'MISSING', f'{p.stat().st_size/1024**3:.2f} GiB' if status else '',name)
print('Ready for template:',ok)

## 06 啟動雲端 ComfyUI
僅綁定 127.0.0.1:8188，不用公開隧道。

In [ ]:
# Start a private ComfyUI server inside Colab. No public tunnel.
import subprocess, sys, time
from pathlib import Path
from urllib.request import urlopen
COMFY='/content/ComfyUI'
LOG='/content/comfyui.log'
def alive():
    try:
        with urlopen('http://127.0.0.1:8188/system_stats', timeout=3) as r:
            return r.status == 200
    except Exception:
        return False
if not alive():
    with open(LOG,'a') as f:
        proc=subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-pinned-memory'],cwd=COMFY,stdout=f,stderr=subprocess.STDOUT,start_new_session=True)
    print('Started PID:',proc.pid)
for _ in range(90):
    if alive():
        print('ComfyUI OK: http://127.0.0.1:8188')
        break
    time.sleep(2)
else:
    print('ComfyUI failed to start. Recent logs:')
    print(Path(LOG).read_text(errors='replace')[-5000:])

## 07 在 Colab 內嵌開啟 ComfyUI
403 錯誤優先使用 iframe；如仍失敗檢查 Colab 帳戶與 Cookie。

In [ ]:
from google.colab import output
# Preferred option if opening a Colab proxy URL in a new tab returns HTTP 403.
output.serve_kernel_port_as_iframe(8188, height=900)

## 08 備份輸出至 Google Drive
執行前請確認 Drive 剩餘儲存額度。

In [ ]:
# Mount Google Drive; the Drive folder consumes Drive quota, not local Windows disk.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil
SOURCE=Path('/content/ComfyUI/output')
TARGET=Path('/content/drive/MyDrive/MiniMax_H3/videos')
TARGET.mkdir(parents=True, exist_ok=True)
copied=0
for p in SOURCE.rglob('*'):
    if not p.is_file():
        continue
    dest=TARGET / p.relative_to(SOURCE)
    dest.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(p,dest)
    copied+=1
print('Backed up',copied,'output files to',TARGET)